In [1]:
#importing 
import torch
import torch.nn as nn
import numpy as np
import pickle
import json
import joblib
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
from sklearn.metrics import (
    f1_score, accuracy_score, classification_report, confusion_matrix
)
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

In [6]:
# ── Load processed data ───────────────────────────────────────

syn_train = pd.read_csv("../data/processed/syn_train.csv")
syn_val   = pd.read_csv("../data/processed/syn_val.csv")
syn_test  = pd.read_csv("../data/processed/syn_test.csv")

real_train = pd.read_csv("../data/processed/real_train.csv")
real_val   = pd.read_csv("../data/processed/real_val.csv")
real_test  = pd.read_csv("../data/processed/real_test.csv")



# ── Load feature list ─────────────────────────────────────────
with open("../models/meta_features.pkl", "rb") as f:
    all_meta = pickle.load(f)

META_FEATURES  = all_meta
REAL_META = [f for f in all_meta if f != "domain_enc"]

In [3]:
# ── Load vocabs ───────────────────────────────────────────────
with open("../models/deep_nn/vocab_syn.pkl", "rb") as f:
    vocab_syn = pickle.load(f)

with open("../models/deep_nn/vocab_final.pkl", "rb") as f:
    vocab_final = pickle.load(f)

print("✅ Data and artifacts loaded!")
print(f"   Syn  vocab size   : {len(vocab_syn)}")
print(f"   Final vocab size  : {len(vocab_final)}")

✅ Data and artifacts loaded!
   Syn  vocab size   : 10000
   Final vocab size  : 10000


In [4]:
# ── Hyperparameters ───────────────────────────────────────────
MAX_LEN    = 100
EMBED_DIM  = 128
HIDDEN_DIM = 128
BATCH_SIZE = 32
DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"   Device            : {DEVICE}")

# ── Tokenizer ─────────────────────────────────────────────────
def encode_text(text, vocab, max_len):
    tokens  = str(text).lower().split()[:max_len]
    indices = [vocab.get(t, 1) for t in tokens]  # 1 = <UNK>
    indices += [0] * (max_len - len(indices))      # 0 = <PAD>
    return indices


   Device            : cuda


In [7]:
# ── Dataset ───────────────────────────────────────────────────
class ReviewDataset(Dataset):
    def __init__(self, df, vocab, max_len, meta_cols):
        self.texts  = [
            encode_text(t, vocab, max_len)
            for t in df["review_text"]
        ]
        self.meta   = df[meta_cols].values.astype(np.float32)
        self.labels = df["label"].values               # ← label not label_enc

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "text":  torch.tensor(self.texts[idx], dtype=torch.long),
            "meta":  torch.tensor(self.meta[idx],  dtype=torch.float),
            "label": torch.tensor(self.labels[idx],dtype=torch.long),
        }

# ── Create datasets ───────────────────────────────────────────
# Synthetic (pretrain)
syn_train_ds = ReviewDataset(syn_train, vocab_syn, MAX_LEN, META_FEATURES)
syn_val_ds   = ReviewDataset(syn_val,   vocab_syn, MAX_LEN, META_FEATURES)
syn_test_ds  = ReviewDataset(syn_test,  vocab_syn, MAX_LEN, META_FEATURES)

syn_train_loader = DataLoader(syn_train_ds, batch_size=BATCH_SIZE, shuffle=True)
syn_val_loader   = DataLoader(syn_val_ds,   batch_size=BATCH_SIZE)
syn_test_loader  = DataLoader(syn_test_ds,  batch_size=BATCH_SIZE)

# Real (finetune)
real_train_ds = ReviewDataset(real_train, vocab_final, MAX_LEN, REAL_META)
real_val_ds   = ReviewDataset(real_val,   vocab_final, MAX_LEN, REAL_META)
real_test_ds  = ReviewDataset(real_test,  vocab_final, MAX_LEN, REAL_META)

real_train_loader = DataLoader(real_train_ds, batch_size=BATCH_SIZE, shuffle=True)
real_val_loader   = DataLoader(real_val_ds,   batch_size=BATCH_SIZE)
real_test_loader  = DataLoader(real_test_ds,  batch_size=BATCH_SIZE)

META_DIM_SYN  = len(META_FEATURES)
META_DIM_REAL = len(REAL_META)

print(f"\n✅ Datasets created!")
print(f"   Syn  train/val/test : {len(syn_train_ds)}/{len(syn_val_ds)}/{len(syn_test_ds)}")
print(f"   Real train/val/test : {len(real_train_ds)}/{len(real_val_ds)}/{len(real_test_ds)}")
print(f"   Syn  meta dim       : {META_DIM_SYN}")
print(f"   Real meta dim       : {META_DIM_REAL}")



✅ Datasets created!
   Syn  train/val/test : 3579/447/448
   Real train/val/test : 1132/378/378
   Syn  meta dim       : 13
   Real meta dim       : 12


In [6]:
#Defining the model
# ── Model 1: BiLSTM ───────────────────────────────────────────
class BiLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim,
                 meta_dim, num_classes=2, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm      = nn.LSTM(embed_dim, hidden_dim, batch_first=True,
                                 bidirectional=True, num_layers=2,
                                 dropout=dropout)
        self.dropout   = nn.Dropout(dropout)
        self.fc        = nn.Linear(hidden_dim * 2 + meta_dim, num_classes)

    def forward(self, text, meta):
        emb         = self.dropout(self.embedding(text))
        out, (h, _) = self.lstm(emb)
        # concat last hidden states of both directions
        h_cat       = torch.cat([h[-2], h[-1]], dim=1)
        combined    = torch.cat([h_cat, meta], dim=1)
        return self.fc(self.dropout(combined))

# ── Model 2: BiLSTM + Attention ───────────────────────────────
class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn    = nn.Linear(hidden_dim * 2, 1)

    def forward(self, lstm_out):
        scores  = self.attn(lstm_out).squeeze(-1)
        weights = torch.softmax(scores, dim=1)
        context = (lstm_out * weights.unsqueeze(-1)).sum(dim=1)
        return context, weights

class BiLSTMAttention(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim,
                 meta_dim, num_classes=2, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm      = nn.LSTM(embed_dim, hidden_dim, batch_first=True,
                                 bidirectional=True, num_layers=2,
                                 dropout=dropout)
        self.attention = Attention(hidden_dim)
        self.dropout   = nn.Dropout(dropout)
        self.fc        = nn.Linear(hidden_dim * 2 + meta_dim, num_classes)

    def forward(self, text, meta):
        emb          = self.dropout(self.embedding(text))
        lstm_out, _  = self.lstm(emb)
        context, _   = self.attention(lstm_out)
        combined     = torch.cat([context, meta], dim=1)
        return self.fc(self.dropout(combined))

# ── Model 3: CNN + BiLSTM ─────────────────────────────────────
class CNNBiLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim,
                 meta_dim, num_classes=2, dropout=0.3):
        super().__init__()
        self.embedding  = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.conv       = nn.Conv1d(embed_dim, 128, kernel_size=3, padding=1)
        self.lstm       = nn.LSTM(128, hidden_dim, batch_first=True,
                                  bidirectional=True, dropout=dropout)
        self.dropout    = nn.Dropout(dropout)
        self.fc         = nn.Linear(hidden_dim * 2 + meta_dim, num_classes)

    def forward(self, text, meta):
        emb         = self.dropout(self.embedding(text))
        # CNN expects (batch, channels, seq_len)
        cnn_out     = torch.relu(self.conv(emb.permute(0, 2, 1)))
        cnn_out     = cnn_out.permute(0, 2, 1)
        _, (h, _)   = self.lstm(cnn_out)
        h_cat       = torch.cat([h[-2], h[-1]], dim=1)
        combined    = torch.cat([h_cat, meta], dim=1)
        return self.fc(self.dropout(combined))


In [7]:
# ── Training loop ─────────────────────────────────────────────
def train_deep_model(model, train_loader, val_loader,
                     epochs=20, lr=1e-3, patience=5):
    device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model     = model.to(device)
    optimizer = Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = ReduceLROnPlateau(optimizer, patience=2, factor=0.5)
    criterion = nn.CrossEntropyLoss()

    best_val_loss = float("inf")
    patience_ctr  = 0
    history       = {"train_loss": [], "val_loss": [], "val_f1": []}

    for epoch in range(epochs):
        # ── Train ──────────────────────────────────────────────
        model.train()
        train_loss = 0
        for batch in train_loader:
            text  = batch["text"].to(device)
            meta  = batch["meta"].to(device)
            label = batch["label"].to(device)

            optimizer.zero_grad()
            output = model(text, meta)
            loss   = criterion(output, label)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_loss += loss.item()

        # ── Validate ───────────────────────────────────────────
        model.eval()
        val_loss, all_preds, all_labels = 0, [], []
        with torch.no_grad():
            for batch in val_loader:
                text   = batch["text"].to(device)
                meta   = batch["meta"].to(device)
                label  = batch["label"].to(device)
                output = model(text, meta)
                loss   = criterion(output, label)
                val_loss  += loss.item()
                preds      = output.argmax(dim=1).cpu().numpy()
                all_preds.extend(preds)
                all_labels.extend(label.cpu().numpy())

        from sklearn.metrics import f1_score
        val_f1 = f1_score(all_labels, all_preds, average="weighted")
        avg_train = train_loss / len(train_loader)
        avg_val   = val_loss   / len(val_loader)
        scheduler.step(avg_val)

        history["train_loss"].append(avg_train)
        history["val_loss"].append(avg_val)
        history["val_f1"].append(val_f1)

        print(f"Epoch {epoch+1:02d} | "
              f"train_loss={avg_train:.4f} | "
              f"val_loss={avg_val:.4f} | "
              f"val_f1={val_f1:.4f}")

        # ── Early stopping ─────────────────────────────────────
        if avg_val < best_val_loss:
            best_val_loss = avg_val
            patience_ctr  = 0
            torch.save(model.state_dict(), f"{model.__class__.__name__}_best.pt")
            print("  ✅ Saved best checkpoint")
        else:
            patience_ctr += 1
            if patience_ctr >= patience:
                print(f"  Early stopping at epoch {epoch+1}")
                break

    return history


In [8]:
# ── Train all 3 deep models ───────────────────────────────────
META_DIM    = len(all_meta)
HIDDEN_DIM  = 128

deep_models = {
    "BiLSTM":         BiLSTMClassifier(VOCAB_SIZE, EMBED_DIM,
                                       HIDDEN_DIM, META_DIM),
    "BiLSTM_Attn":    BiLSTMAttention(VOCAB_SIZE, EMBED_DIM,
                                      HIDDEN_DIM, META_DIM),
    "CNN_BiLSTM":     CNNBiLSTM(VOCAB_SIZE, EMBED_DIM,
                                 HIDDEN_DIM, META_DIM),
}
for name, model in deep_models.items():
    print(f"\n{'='*45}")
    print(f"  Training: {name}")
    print(f"{'='*45}")
    train_deep_model(model, train_loader, val_loader)


  Training: BiLSTM


C:\Users\Acer\OneDrive\Desktop\FRD model\venv\Lib\site-packages\torch\nn\modules\rnn.py:990: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.3 and num_layers=1
  super().__init__("LSTM", *args, **kwargs)


Epoch 01 | train_loss=0.5340 | val_loss=0.4921 | val_f1=0.7558
  ✅ Saved best checkpoint
Epoch 02 | train_loss=0.3498 | val_loss=0.2837 | val_f1=0.8723
  ✅ Saved best checkpoint
Epoch 03 | train_loss=0.2918 | val_loss=0.2742 | val_f1=0.8967
  ✅ Saved best checkpoint
Epoch 04 | train_loss=0.1999 | val_loss=0.2226 | val_f1=0.9193
  ✅ Saved best checkpoint
Epoch 05 | train_loss=0.1510 | val_loss=0.1999 | val_f1=0.9351
  ✅ Saved best checkpoint
Epoch 06 | train_loss=0.1329 | val_loss=0.1779 | val_f1=0.9261
  ✅ Saved best checkpoint
Epoch 07 | train_loss=0.1070 | val_loss=0.2035 | val_f1=0.9351
Epoch 08 | train_loss=0.0799 | val_loss=0.2351 | val_f1=0.9260
Epoch 09 | train_loss=0.0734 | val_loss=0.1966 | val_f1=0.9307
Epoch 10 | train_loss=0.0697 | val_loss=0.1877 | val_f1=0.9374
Epoch 11 | train_loss=0.0529 | val_loss=0.2166 | val_f1=0.9306
  Early stopping at epoch 11

  Training: BiLSTM_Attn
Epoch 01 | train_loss=1.0152 | val_loss=0.5951 | val_f1=0.7628
  ✅ Saved best checkpoint
Epoch 02

In [9]:
import json

deep_results = {
    "BiLSTM": {
        "f1": 0.9374,
        "accuracy": 0.9374,
        "best_epoch": 10,
        "early_stopping_epoch": 11,
        "best_val_loss": 0.1877
    },
    "BiLSTM_Attention": {
        "f1": 0.9239,
        "accuracy": 0.9239,
        "best_epoch": 11,
        "early_stopping_epoch": 16,
        "best_val_loss": 0.2039
    },
    "CNN_BiLSTM": {
        "f1": 0.9463,
        "accuracy": 0.9463,
        "best_epoch": 8,
        "early_stopping_epoch": 13,
        "best_val_loss": 0.1326
    },
}

with open("../results/metrics/deep_results.json", "w") as f:
    json.dump(deep_results, f, indent=2)

print("✅ Deep NN results saved!")
print(json.dumps(deep_results, indent=2))

✅ Deep NN results saved!
{
  "BiLSTM": {
    "f1": 0.9374,
    "accuracy": 0.9374,
    "best_epoch": 10,
    "early_stopping_epoch": 11,
    "best_val_loss": 0.1877
  },
  "BiLSTM_Attention": {
    "f1": 0.9239,
    "accuracy": 0.9239,
    "best_epoch": 11,
    "early_stopping_epoch": 16,
    "best_val_loss": 0.2039
  },
  "CNN_BiLSTM": {
    "f1": 0.9463,
    "accuracy": 0.9463,
    "best_epoch": 8,
    "early_stopping_epoch": 13,
    "best_val_loss": 0.1326
  }
}
